In [ ]:
# ============================================================
# Cell 1: 安装依赖 + 启动 Ollama (GPU) + 构建 RAG
# ✅ 新增: FAISS 缓存 (保存/加载磁盘) + folder 元数据
# ============================================================

import subprocess, os, sys

# --- 检测运行环境 ---
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f'📍 运行环境: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "其他"}')

# --- GPU 检测 ---
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU 已连接: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU! 请在 Kaggle 右侧 Settings → Accelerator 选择 GPU')
    print('   或在 Colab: Runtime → Change runtime type → GPU')

# --- 系统依赖 ---
!apt-get update -qq && apt-get install -y -qq zstd ffmpeg > /dev/null 2>&1

# --- Python 依赖 ---
!pip install -q openai-whisper edge-tts aiohttp gradio langchain langchain-community faiss-cpu sentence-transformers

# --- 安装 Ollama ---
!curl -fsSL https://ollama.com/install.sh | sh

# --- 启动 Ollama (GPU 模式) ---
import time, threading

def start_ollama():
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    env['OLLAMA_ORIGINS'] = '*'
    env['OLLAMA_GPU_LAYERS'] = '999'
    subprocess.Popen(['ollama', 'serve'], env=env)

threading.Thread(target=start_ollama, daemon=True).start()
print('⏳ 等待 Ollama 启动...')

import requests
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 (耗时 {i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ Ollama 启动超时，请检查日志')

print('Downloading LLM models...')
PRIMARY_MODEL = 'llama3.1'
FALLBACK_MODEL = 'llama3.2'

def pull_model(model_name):
    print(f'Trying to pull model: {model_name}')
    return subprocess.run(['ollama', 'pull', model_name], check=False).returncode == 0

if pull_model(PRIMARY_MODEL):
    ACTIVE_MODEL = PRIMARY_MODEL
else:
    print(f'Warning: cannot use {PRIMARY_MODEL}, falling back to {FALLBACK_MODEL}')
    if not pull_model(FALLBACK_MODEL):
        raise RuntimeError('Model download failed. Please check Ollama network and disk space.')
    ACTIVE_MODEL = FALLBACK_MODEL

print(f'Active model: {ACTIVE_MODEL}')

!echo '---'; curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; d=json.load(sys.stdin); [print(f'  模型: {m[\"name\"]}') for m in d.get('models',[])]"

# --- RAG 知识库构建 ---
import glob
from pathlib import Path

try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ✅ [新增] FAISS 缓存路径
FAISS_PATH = os.path.join(WORK_DIR, 'faiss_index')

# ✅ 自动检测知识库路径 (兼容 Colab / Kaggle)
possible_kb_paths = [
    '/kaggle/input/knowledge-base/RAG/data',
    '/kaggle/input',
    '/content/drive/MyDrive/Colab Notebooks/RAG/data/knowledge_base',
    os.path.join(WORK_DIR, 'knowledge_base'),
]

# Colab: 挂载 Drive
if IS_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
    except:
        pass

def load_hwu_documents(root_dir):
    """加载文档，✅ 新增 folder 元数据用于文件夹路由 RAG"""
    docs = []
    md_files = glob.glob(os.path.join(root_dir, '**/*.md'), recursive=True)
    print(f'  扫描: {root_dir} → {len(md_files)} 个 .md 文件')
    for fp in md_files:
        try:
            text = Path(fp).read_text(encoding='utf-8')
            lines = text.split('\n', 2)
            if len(lines) < 3:
                continue
            source = lines[0].replace('Source:', '').strip()
            topic = lines[1].replace('Topic:', '').strip()
            content = lines[2].strip()

            # ✅ [新增] 提取文件夹名作为 folder 元数据
            folder = fp.split('knowledge_base/')[-1].split('/')[0] \
                     if 'knowledge_base' in fp else topic

            docs.append(Document(
                page_content=content,
                metadata={
                    'source': source,
                    'topic': topic,
                    'filename': Path(fp).name,
                    'folder': folder,        # ✅ [新增] 供 Cell 2 文件夹路由使用
                }
            ))
        except Exception as e:
            print(f'  ⚠️ {fp}: {e}')
    return docs

# ✅ [新增] FAISS 缓存: 如果已有索引则直接加载，跳过重复构建
vector_db = None
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cuda' if HAS_GPU else 'cpu'}
)

if os.path.exists(FAISS_PATH):
    print('📂 加载已有 FAISS 向量库...')
    vector_db = FAISS.load_local(
        FAISS_PATH, embeddings,
        allow_dangerous_deserialization=True
    )
    print('✅ 向量库从缓存加载完成')
else:
    for kb_path in possible_kb_paths:
        if os.path.exists(kb_path):
            documents = load_hwu_documents(kb_path)
            if documents:
                chunks = RecursiveCharacterTextSplitter(
                    chunk_size=1000, chunk_overlap=200
                ).split_documents(documents)
                print(f'  切分为 {len(chunks)} 个片段，构建向量索引...')
                vector_db = FAISS.from_documents(chunks, embeddings)
                # ✅ [新增] 保存到磁盘，下次直接加载
                vector_db.save_local(FAISS_PATH)
                print(f'  ✅ RAG 向量库构建完成 ({len(chunks)} chunks)，已保存到 {FAISS_PATH}')
                break

if vector_db is None:
    print('⚠️ 未找到知识库，将以纯 LLM 模式运行')
    print(f'  Kaggle: 请上传 .md 文件到 Dataset')
    print(f'  Colab: 请确认 Google Drive 中有知识库')

print('\n🎉 所有准备工作完成！请运行下一个 Cell 启动语音助手。')

📍 运行环境: Kaggle
✅ GPU 已连接: Tesla P100-PCIE-16GB (15.9 GB)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
⏳ 等待 Ollama 启动...


time=2026-03-11T15:45:17.479Z level=INFO source=routes.go:1658 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[* http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_RE

[GIN] 2026/03/11 - 15:45:18 | 200 |     625.915µs |             ::1 | GET      "/api/tags"
✅ Ollama 启动成功 (耗时 2s)
📥 下载 LLM 模型...
]11;?\[GIN] 2026/03/11 - 15:45:23 | 200 |      75.099µs |       127.0.0.1 | HEAD     "/"
pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ [GIN] 2026/03/11 - 15:45:24 | 200 |   512.44759ms |       127.0.0.1 | POST     "/api/pull"
pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 
✅ LLM 模型就绪
[GIN] 2026/03/11 - 15:45:24 | 200 

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📂 加载已有 FAISS 向量库...
✅ 向量库从缓存加载完成

🎉 所有准备工作完成！请运行下一个 Cell 启动语音助手。


In [33]:
import subprocess, os, time, requests

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
subprocess.Popen(['ollama', 'serve'])

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 ({i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ 启动失败，查看日志:')
    !cat /tmp/ollama.log 2>/dev/null || echo "无日志"

time=2026-03-11T16:26:01.816Z level=INFO source=routes.go:1658 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_DEBUG:INFO OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://0.0.0.0:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[* http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_RE

[GIN] 2026/03/11 - 16:26:02 | 200 |     693.847µs |             ::1 | GET      "/api/tags"
✅ Ollama 启动成功 (2s)


In [ ]:
# ============================================================
# Cell 2: 实时语音通话 (优化版: 低延迟 + TTS 修复 + GPU 加速)
# ✅ 新增: LLM预热 | 文件夹路由RAG | 流式输出 | TTS异步分离
# ✅ 合并: 多句并行TTS | Link Protocol | 心理健康关键词扩展
# ============================================================

import gradio as gr
import json
import requests
import whisper
import edge_tts
import asyncio
import os
import time
import re
import tempfile
import shutil
import subprocess
import numpy as np
import scipy.io.wavfile as wavfile
import torch
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

# ✅ 关键修复: 允许嵌套事件循环 (Kaggle/Jupyter 自带 loop)
import nest_asyncio
nest_asyncio.apply()

# ============================================================
# Whisper 加载 (GPU)
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Whisper 设备: {device}')

WHISPER_MODEL_SIZE = 'base'
stt_model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)

if device == 'cuda':
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (GPU)')
else:
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (CPU - 会较慢)')

# ============================================================
# ✅ [新增 #8] LLM 后台预热
# 启动时立刻在后台发一个 hi 请求，提前把模型加载到 GPU
# ============================================================
_llm_ready = threading.Event()

def _warmup():
    model_name = globals().get('ACTIVE_MODEL', 'llama3.2')
    print(f'Warming up LLM in background: {model_name}')
    for attempt in range(3):
        try:
            r = requests.post('http://localhost:11434/api/chat', json={
                'model': model_name,
                'messages': [{'role':'user','content':'hi'}],
                'stream': False,
                'keep_alive': '60m',
                'options': {'num_predict': 3, 'num_ctx': 256}
            }, timeout=30)
            if r.status_code == 200:
                _llm_ready.set()
                print(f'Background warmup completed: {model_name}')
                return
        except Exception as e:
            print(f'Warmup attempt {attempt+1} failed: {e}')
            time.sleep(5)
    print(f'LLM warmup failed: {model_name}')

threading.Thread(target=_warmup, daemon=True).start()

# ============================================================
# VAD 参数
# ============================================================
SILENCE_THRESHOLD = 0.012
SILENCE_DURATION  = 1.2
MIN_SPEECH_DURATION = 0.3

def detect_speech(audio_chunk, sr):
    if audio_chunk is None or len(audio_chunk) == 0:
        return False
    if audio_chunk.dtype == np.int16:
        audio_float = audio_chunk.astype(np.float32) / 32768.0
    elif audio_chunk.dtype == np.int32:
        audio_float = audio_chunk.astype(np.float32) / 2147483648.0
    else:
        audio_float = audio_chunk.astype(np.float32)
    rms = np.sqrt(np.mean(audio_float ** 2))
    return rms > SILENCE_THRESHOLD

def save_buffer_to_wav(audio_buffer, sr):
    if not audio_buffer:
        return None
    combined = np.concatenate(audio_buffer)
    if len(combined) < sr * MIN_SPEECH_DURATION:
        return None
    path = os.path.join(tempfile.gettempdir(), f'voice_{int(time.time()*1000)}.wav')
    if combined.dtype != np.int16:
        if np.max(np.abs(combined)) <= 1.0:
            combined = (combined * 32767).astype(np.int16)
        else:
            combined = combined.astype(np.int16)
    wavfile.write(path, sr, combined)
    return path

# ============================================================
# ASR
# ============================================================
def speech_to_text(audio_path):
    if not audio_path:
        return ''
    try:
        t0 = time.time()
        result = stt_model.transcribe(
            audio_path,
            language='en',
            fp16=(device == 'cuda'),
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            compression_ratio_threshold=2.4,
        )
        text = result['text'].strip()
        elapsed = time.time() - t0

        hallucinations = {
            'thank you', 'thanks for watching', 'subscribe',
            'you', 'bye', 'the end', '字幕', '感谢观看',
            '请不吝点赞', '订阅', '谢谢大家', ''
        }
        if text.lower().strip('.!? ') in hallucinations:
            return ''

        print(f'🎤 识别 ({elapsed:.2f}s): {text}')
        return text
    except Exception as e:
        print(f'❌ ASR Error: {e}')
        return ''

# ============================================================
# TTS (合并改进: WAV输出 + 重试3次 + 文本清理抽取)
# ============================================================
def _run_tts_sync(text, voice, output_path, retries=3):
    """将文本转为 WAV 音频文件 (✅ 重试次数 2→3)"""
    for attempt in range(retries):
        try:
            mp3_path = output_path.replace('.wav', '.mp3')
            loop = asyncio.get_event_loop()
            communicate = edge_tts.Communicate(text, voice, rate='+5%')
            loop.run_until_complete(communicate.save(mp3_path))

            if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 500:
                subprocess.run(
                    ['ffmpeg', '-y', '-loglevel', 'error',
                     '-i', mp3_path, '-ar', '22050', '-ac', '1', output_path],
                    timeout=15
                )
                try: os.remove(mp3_path)
                except: pass
                if os.path.exists(output_path) and os.path.getsize(output_path) > 500:
                    return True
            else:
                print(f'  ⚠️ TTS 文件太小 (attempt {attempt+1})')
        except Exception as e:
            print(f'  ⚠️ TTS attempt {attempt+1}: {e}')
            time.sleep(0.3)
    return False

def _clean_text_for_tts(text):
    """清理文本使其适合 TTS 朗读"""
    clean = text.split('---')[0]
    for pattern, repl in [
        (r'\(https?://[^\)]+\)', ''),
        (r'https?://\S+', ''),
        (r'For more information,?\s*visit:?\s*', ''),
        (r'[*#_>~`\[\](){}|]', ''),
        (r'\\n|\\r', ' '),
        (r'\n+', '. '),
        (r'\s{2,}', ' '),
        (r'\.{2,}', '.'),
    ]:
        clean = re.sub(pattern, repl, clean)
    return clean.strip()

def _get_tts_voice(text):
    """根据文本语言选择 TTS 语音"""
    has_cjk = bool(re.search(r'[\u4e00-\u9fff]', text))
    return 'zh-CN-XiaoxiaoNeural' if has_cjk else 'en-US-EmmaNeural'

def text_to_speech(text, voice=None):
    """完整文本 → WAV 音频。voice=None 则自动检测语言。"""
    if not text or len(text.strip()) < 2:
        return None

    clean = _clean_text_for_tts(text)
    if not clean or len(clean) < 2:
        return None

    if len(clean) > 1500:
        cut = clean[:1500].rfind('.')
        clean = clean[:cut+1] if cut > 100 else clean[:1500]

    v = voice or _get_tts_voice(clean)
    path = os.path.join(tempfile.gettempdir(), f'tts_{int(time.time()*1000)}.wav')

    if _run_tts_sync(clean, v, path):
        print(f'🔊 TTS OK: {len(clean)} chars')
        return path
    else:
        print('❌ TTS 最终失败')
        return None

# ============================================================
# ✅ 强制 Gradio 每次都重新播放音频
# ============================================================
def make_unique_audio(audio_path):
    if audio_path is None:
        return None
    try:
        ext = os.path.splitext(audio_path)[1]
        unique_path = os.path.join(tempfile.gettempdir(), f'play_{int(time.time()*1000)}{ext}')
        shutil.copy2(audio_path, unique_path)
        return unique_path
    except Exception as e:
        print(f'⚠️ copy error: {e}')
        return audio_path

# ============================================================
# ✅ 音频拼接工具 (从改进版合并)
# ============================================================
def concat_wav_files(wav_paths):
    """用 ffmpeg 拼接多个 WAV 文件为一个完整音频。
    跳过 None 和不存在的文件。返回拼接后的文件路径或 None。"""
    valid = [p for p in wav_paths if p and os.path.exists(p) and os.path.getsize(p) > 500]
    if not valid:
        return None
    if len(valid) == 1:
        return valid[0]
    
    list_path = os.path.join(tempfile.gettempdir(), f'concat_{int(time.time()*1000)}.txt')
    out_path  = os.path.join(tempfile.gettempdir(), f'joined_{int(time.time()*1000)}.wav')
    
    try:
        with open(list_path, 'w') as f:
            for p in valid:
                f.write(f"file '{p}'\n")
        
        result = subprocess.run(
            ['ffmpeg', '-y', '-loglevel', 'error',
             '-f', 'concat', '-safe', '0', '-i', list_path,
             '-ar', '22050', '-ac', '1', out_path],
            timeout=15, capture_output=True
        )
        
        try: os.remove(list_path)
        except: pass
        
        if result.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 500:
            print(f'🔗 拼接 {len(valid)} 段音频 → {out_path}')
            return out_path
        else:
            print(f'⚠️ 拼接失败: {result.stderr.decode()[:100]}')
            return valid[0]
    except Exception as e:
        print(f'⚠️ 拼接异常: {e}')
        return valid[0]

# ============================================================
# ✅ 多句并行 TTS 工具函数 (从改进版合并)
# ============================================================
_MIN_SENTENCE_LEN = 30

def tts_single_sentence(text, index, voice='en-US-EmmaNeural'):
    """对单个句子生成 TTS 音频。返回 (index, audio_path)。"""
    if not text or len(text.strip()) < 2:
        return (index, None)
    clean = _clean_text_for_tts(text)
    if not clean or len(clean) < 2:
        return (index, None)
    v = _get_tts_voice(clean) if voice is None else voice
    path = os.path.join(tempfile.gettempdir(), f'tts_chunk_{index}_{int(time.time()*1000)}.wav')
    if _run_tts_sync(clean, v, path):
        return (index, path)
    return (index, None)

# ============================================================
# ✅ [新增 #9] 文件夹路由 RAG (合并: 心理健康关键词扩展)
# ============================================================
FOLDER_ROUTER = {
    'accommodation': ['accommodation'],
    'hall': ['accommodation'],
    'housing': ['accommodation'],
    'room': ['accommodation'],
    'rent': ['accommodation'],
    'hostel': ['accommodation'],
    'fee': ['finance'],
    'tuition': ['finance'],
    'scholarship': ['finance'],
    'bursary': ['finance'],
    'funding': ['finance'],
    'loan': ['finance'],
    'money': ['finance'],
    'pay': ['finance'],
    'council tax': ['finance'],
    'enrol': ['enrollment'],
    'enroll': ['enrollment'],
    'register': ['enrollment'],
    'registration': ['enrollment'],
    'id card': ['enrollment'],
    'student card': ['enrollment'],
    'induction': ['enrollment'],
    'course': ['courses'],
    'module': ['courses'],
    'programme': ['courses'],
    'timetable': ['courses'],
    'lecture': ['courses'],
    'class': ['courses'],
    'major': ['courses'],
    'degree': ['courses'],
    'study': ['courses'],
    'exam': ['exams_assessment'],
    'assessment': ['exams_assessment'],
    'grade': ['exams_assessment'],
    'extension': ['exams_assessment'],
    'resit': ['exams_assessment'],
    'mental health': ['support'],
    'wellbeing': ['support'],
    'counsell': ['support'],
    'disability': ['support'],
    'welfare': ['support'],
    'anxiety': ['support'],
    'stress': ['support'],
    # ✅ 合并: 心理健康关键词扩展
    'depress': ['support'],
    'lonely': ['support'],
    'homesick': ['support'],
    'unhappy': ['support'],
    'sad': ['support'],
    'overwhelm': ['support'],
    'struggle': ['support'],
    'crisis': ['support'],
    'sport': ['facilities'],
    'gym': ['facilities'],
    'oriam': ['facilities'],
    'library': ['facilities'],
    'canteen': ['facilities'],
    'food': ['facilities'],
    'cafe': ['facilities'],
    'parking': ['facilities'],
    'visa': ['support', 'enrollment'],
    'international': ['enrollment', 'support'],
    'career': ['support'],
    'job': ['support'],
    'club': ['support'],
    'society': ['support'],
    'union': ['support'],
}

def get_target_folders(query):
    """根据问题关键词快速定位相关文件夹"""
    q = query.lower()
    folders = set()
    for kw, flds in FOLDER_ROUTER.items():
        if kw in q:
            folders.update(flds)
    return list(folders) if folders else None  # None 表示全库搜索

def user_wants_link(query):
    """检测用户是否在询问链接/网址"""
    q = query.lower()
    link_keywords = ['link', 'url', 'website', 'web site', 'webpage', 'web page',
                     'address', 'where can i find', 'where do i find',
                     'send me the link', 'give me the link', 'share the link',
                     'how to access', 'how do i access', 'portal',
                     '链接', '网址', '网站']
    return any(kw in q for kw in link_keywords)

def is_verified_url(raw_value):
    if not raw_value or not isinstance(raw_value, str):
        return False
    value = raw_value.strip()
    if not re.match(r'^https?://\S+$', value):
        return False
    parsed = urlparse(value)
    return parsed.scheme in {'http', 'https'} and bool(parsed.netloc)

def get_verified_url(raw_value):
    return raw_value.strip() if is_verified_url(raw_value) else ''

def enforce_real_links(text, rag_links):
    """
    智能链接拦截器（双杀版）：
    1. 拦截假的 Markdown 链接 [Text](URL) -> 扒掉变成 Text
    2. 拦截假的裸露链接 https://... -> 直接删除
    """
    import re
    valid_urls = []
    for lnk in rag_links:
        if ': ' in lnk:
            candidate_url = lnk.split(': ', 1)[-1].strip()
        else:
            candidate_url = lnk.strip()
        vu = get_verified_url(candidate_url)
        vu = re.sub(r'^https?://', '', vu.lower()).rstrip('/') if vu else ''
        if vu:
            valid_urls.append(vu)
            
    def is_url_valid(test_url):
        verified_url = get_verified_url(test_url)
        if not verified_url:
            return False
        lu_norm = re.sub(r'^https?://', '', verified_url.lower()).rstrip('/')
        # 必须完全一致，或者带有 # 锚点
        return any(lu_norm == vu or lu_norm.startswith(vu + '#') for vu in valid_urls)

    # 1. 替换 Markdown 格式: [Text](URL)
    def md_replacer(match):
        link_text = match.group(1)
        link_url = match.group(2)
        if is_url_valid(link_url):
            return match.group(0)  # 合法，放行
        return link_text           # 不合法，只留文字

    text = re.sub(r'\[([^\]]+)\]\((https?://[^\)]+)\)', md_replacer, text)

    # 2. 替换裸露的 URL 格式: https://... (防止 LLM 不听话直接输出链接)
    def raw_replacer(match):
        link_url = match.group(0)
        if is_url_valid(link_url):
            return link_url  # 合法，放行
        return ""            # 不合法，直接删除这个假链接

    text = re.sub(r'https?://[a-zA-Z0-9./\-_#?=&]+', raw_replacer, text)
    
    # 清理因为删除裸露链接导致的空冒号，比如 "activities: \n" 变成 "activities.\n"
    text = re.sub(r':\s*(?=\n|$)', '.', text)

    return text.strip()

def retrieve_context(query):
    """返回 (context_text, source_links)"""
    if 'vector_db' not in globals() or vector_db is None:
        return '', []
    try:
        t0 = time.time()
        target_folders = get_target_folders(query)

        if target_folders:
            candidates = vector_db.as_retriever(
                search_kwargs={'k': 15}
            ).invoke(query)
            docs = [d for d in candidates
                    if d.metadata.get('folder','') in target_folders][:5]
            if not docs:
                docs = candidates[:5]
        else:
            docs = vector_db.as_retriever(
                search_kwargs={'k': 5}
            ).invoke(query)

        parts = []
        seen_links = set()
        source_links_raw = []
        for i, d in enumerate(docs):
            raw_source_url = str(d.metadata.get('source', '')).strip()
            verified_source_url = get_verified_url(raw_source_url)
            topic = str(d.metadata.get('topic', 'N/A')).strip() or 'N/A'
            # 收集去重的有效链接
            if verified_source_url and verified_source_url not in seen_links:
                seen_links.add(verified_source_url)
                # 计算链接与查询的相关性分数：URL路径中包含查询关键词的优先
                q_words = set(query.lower().split())
                url_lower = verified_source_url.lower()
                score = sum(1 for w in q_words if w in url_lower and len(w) > 2)
                # 路径越短（越接近主页面）得分越高
                path_depth = max(verified_source_url.count('/') - 2, 0)  # 减去 https:// 的两个斜杠
                score -= path_depth * 0.1
                # 从 URL 路径提取更有意义的标题
                url_path = verified_source_url.rstrip('/').split('/')[-1].replace('-', ' ').title() or topic or 'Link'
                source_links_raw.append((score, f'{url_path}: {verified_source_url}'))
            source_label = verified_source_url if verified_source_url else 'No verified URL'
            parts.append(
                f'--- Document {i} ---\n'
                f'Topic: {topic}\n'
                f'Source URL: {source_label}\n'
                f'Content: {d.page_content}'
            )
        # 按相关性排序，最相关的在前
        source_links_raw.sort(key=lambda x: x[0], reverse=True)
        source_links = [item[1] for item in source_links_raw]
        elapsed = time.time() - t0
        folders_str = str(target_folders) if target_folders else 'all'
        print(f'🔍 RAG ({elapsed:.2f}s) folders={folders_str} → {len(docs)} docs, {len(source_links)} links')
        return '\n\n'.join(parts), source_links
    except Exception as e:
        print(f'RAG Error: {e}')
        return '', []

# ============================================================
# ✅ LLM 调用 — generator 流式输出
# ✅ 合并: System Prompt Link Protocol (禁止编造URL)
# ============================================================
def call_llm(user_text, chat_history):
    # ✅ [新增 #8] 等待预热完成
    if not _llm_ready.is_set():
        _llm_ready.wait(timeout=120)
        if not _llm_ready.is_set():
            yield 'Still loading, please wait a moment.'
            return

    # ✅ 修复: 避免多轮上下文严重污染当前检索
    # 只有当用户输入极短（追问），且没有明确开启新话题时，才拼接上一轮问题
    recent = (chat_history or [])[-1:]
    is_short_follow_up = len(user_text.split()) <= 6 or 'link' in user_text.lower().strip()
    
    user_text_lower = user_text.lower()
    # 只要用户提到了明确的新话题（如 facilities, accommodation），就坚决不拼接上一轮问题
    if recent and recent[0][0] and is_short_follow_up and 'facilities' not in user_text_lower and 'accommodation' not in user_text_lower:
        augmented_query = recent[0][0] + ' ' + user_text
    else:
        augmented_query = user_text
        
    ctx, _rag_links = retrieve_context(augmented_query)
    prompt = f"""
    You are the AI Student Services Assistant for Heriot-Watt University (HWU).
    You must be highly empathetic, professional, and supportive.

    Here is the retrieved knowledge from the university database:
    [CONTEXT START]
    {ctx}
    [CONTEXT END]

    You must follow these prioritized guidelines:

    **CRITICAL Link Protocol (YOU MUST FOLLOW STRICTLY):**
    - NO HALLUCINATION: You MUST ONLY answer using the exact facts provided in the [CONTEXT]. Do NOT use your general knowledge.
    - INLINE LINKS: When mentioning a facility or service, embed its link directly in your text using Markdown format: [Description](Source URL).
    - MATCHING RULE (IMPORTANT): Ensure the `Source URL` logically matches the text. If the [CONTEXT] only provides a general `/facilities` URL, do NOT format it as a fake specific website for a building (e.g. do NOT write [Oriam](.../facilities)). Instead, say "You can find more about Oriam on the [Campus Facilities page](.../facilities)."
    - EXACT URLS ONLY: ONLY use the exact 'Source URL' explicitly provided in the [CONTEXT]. NEVER invent, shorten, or truncate URLs.
    - If no verified URL appears in the [CONTEXT], do not include any link.
    - HANDLING LINK REQUESTS: Even if the user explicitly asks for links, DO NOT create a repetitive bulleted list of the same URL. Write a natural paragraph explaining where to find the information using the provided 'Source URL'.

    **Priority 1: Crisis Intervention & Mental Health**
    If the user expresses distress, depression, anxiety, or a crisis:
    - Respond with deep empathy and humanistic care.
    - Validate their feelings and prioritize their safety.
    - IMMEDIATELY suggest they contact HWU Student Wellbeing Services.
    - Do NOT say "I apologize, I can only answer...". Be a supportive human.

    **Priority 2: Daily Conversation & Greetings**
    If the user says hello, asks how you are, or engages in basic small talk:
    - Respond warmly and naturally.
    - Gently guide the conversation back to how you can help them with Heriot-Watt University student services.

    **Priority 3: HWU Student Services Queries**
    If the question is about HWU (accommodation, enrollment, campus life, etc.):
    - Provide a RICH, detailed, and structured answer using the [CONTEXT].
    - Expand on the details to make the answer comprehensive rather than overly brief.
    - If any LINK in the [CONTEXT] is relevant, naturally weave it into your answer.
    - Pay close attention to ALL documents retrieved — important details like URLs, dates, and contact info may appear in any of them.

    **Priority 4: Out of Scope Queries**
    If the question is completely unrelated to HWU or student services (e.g., coding, math, global news):
    - Politely decline by saying: "I'd love to chat about that, but my expertise is focused on Heriot-Watt University Student Services. How can I help you with your university life today?"

    Output Guidelines:
    - ALWAYS reply in the same language the user is speaking (e.g., if asked in Chinese, reply in natural Chinese).
    - Maintain a conversational and rich tone.
    """

    msgs = [{'role': 'system', 'content': prompt}]
    recent_history = (chat_history or [])[-3:]
    for u, a in recent_history:
        msgs.append({'role': 'user', 'content': u})
        if a:
            msgs.append({'role': 'assistant', 'content': a})
    msgs.append({'role': 'user', 'content': user_text})

    try:
        t0 = time.time()
        model_name = globals().get('ACTIVE_MODEL', 'llama3.2')
        resp = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': model_name,
                'messages': msgs,
                'stream': True,              # ✅ 流式
                'keep_alive': '60m',
                'options': {
                    'num_predict': 400,
                    'temperature': 0.6,
                    'num_ctx': 2048,
                }
            },
            timeout=120,
            stream=True
        )

        # ✅ 流式 yield: 每个 token 立即推送
        full_text = ''
        for line in resp.iter_lines():
            if line:
                try:
                    chunk = json.loads(line)
                    token = chunk.get('message',{}).get('content','')
                    if token:
                        full_text += token
                        yield full_text      # ← 每次 yield 累积文本
                    if chunk.get('done', False):
                        break
                except json.JSONDecodeError:
                    continue

        elapsed = time.time() - t0
        print(f'🤖 LLM ({elapsed:.1f}s): {full_text[:100]}...')
        
        # ✅ 使用代码级硬拦截，剔除所有幻觉链接，只保留纯正文或真实的内嵌链接
        if full_text:
            full_text = enforce_real_links(full_text, _rag_links)
            yield full_text

        if not full_text:
            yield 'Sorry, I could not generate a response.'

    except requests.Timeout:
        yield 'Sorry, the response timed out. Please try again.'
    except Exception as e:
        print(f'❌ LLM Error: {e}')
        yield f'Sorry, service error: {e}'

# ============================================================
# ✅ 多句并行 TTS 流水线 (从改进版合并)
#
# 原理:
#   LLM 流式输出: [===句1===][===句2===][===句3===]...
#   TTS 句1:       [==TTS==]                           ← 与 LLM 并行
#   TTS 句2:                  [==TTS==]                ← 与 LLM 并行
#   TTS 句3:                              [==TTS==]   ← LLM结束后可能已完成
#   拼接:                                          [concat] → 完整音频
# ============================================================
def stream_llm_with_parallel_tts(user_text, chat_history):
    """
    Stream LLM output sentence-by-sentence.
    Each completed sentence is immediately submitted to TTS in a thread pool.
    After LLM finishes, collect all TTS audio in order and concatenate.
    Returns: (full_text, audio_path)
    """
    full_text = ''
    current_buffer = ''
    sentence_index = 0
    tts_executor = ThreadPoolExecutor(max_workers=3)
    tts_futures = []

    voice = _get_tts_voice(user_text)

    for partial in call_llm(user_text, chat_history):
        new_tokens = partial[len(full_text):]
        full_text = partial
        current_buffer += new_tokens

        if (re.search(r'[.!?。！？]\s*$', current_buffer)
                and len(current_buffer.strip()) >= _MIN_SENTENCE_LEN):
            sentence = current_buffer.strip()
            future = tts_executor.submit(tts_single_sentence, sentence, sentence_index, voice)
            tts_futures.append((sentence_index, future))
            print(f'  📝 Chunk {sentence_index}: "{sentence[:60]}..." → TTS submitted')
            sentence_index += 1
            current_buffer = ''

    # Flush remaining text
    if current_buffer.strip() and len(current_buffer.strip()) >= 2:
        future = tts_executor.submit(tts_single_sentence, current_buffer.strip(), sentence_index, voice)
        tts_futures.append((sentence_index, future))
        print(f'  📝 Chunk {sentence_index} (final): "{current_buffer.strip()[:60]}..."')

    if not full_text:
        tts_executor.shutdown(wait=False)
        return 'Sorry, I could not generate a response.', None

    print(f'🤖 LLM done, {len(tts_futures)} TTS chunks, waiting for remaining TTS...')

    # Collect TTS results IN ORDER
    tts_futures.sort(key=lambda x: x[0])
    audio_paths = []
    for idx, future in tts_futures:
        try:
            _, audio_path = future.result(timeout=30)
            if audio_path:
                audio_paths.append(audio_path)
        except Exception as e:
            print(f'  ⚠️ TTS chunk {idx} failed: {e}')

    tts_executor.shutdown(wait=False)

    final_audio = concat_wav_files(audio_paths)

    # ✅ 清理临时 chunk 文件
    for p in audio_paths:
        if p and p != final_audio:
            try: os.remove(p)
            except: pass

    return full_text, final_audio

# ============================================================
# 核心: 流式音频 VAD 回调 (✅ 使用多句并行TTS + 临时文件清理)
# ============================================================
def process_streaming_audio(audio_chunk, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }

    if state_data.get('processing', False):
        return state_data, state_data.get('chat_history', []), gr.update(), '🔄 AI is processing...'

    if audio_chunk is None:
        # ✅ [新增 #8] 显示预热状态
        s = '🎙️ Waiting for you to speak...' if _llm_ready.is_set() else '⏳ LLM loading...'
        return state_data, state_data.get('chat_history', []), gr.update(), s

    sr, audio_data = audio_chunk
    if len(audio_data.shape) > 1:
        audio_data = audio_data[:, 0]

    now = time.time()
    is_voice = detect_speech(audio_data, sr)

    if is_voice:
        state_data['audio_buffer'].append(audio_data)
        state_data['is_speaking'] = True
        state_data['speech_detected'] = True
        state_data['silence_start'] = None
        return state_data, state_data.get('chat_history', []), gr.update(), '🗣️ Listening...'
    else:
        if state_data['speech_detected']:
            state_data['audio_buffer'].append(audio_data)
            if state_data['silence_start'] is None:
                state_data['silence_start'] = now

            silence_elapsed = now - state_data['silence_start']

            if silence_elapsed >= SILENCE_DURATION:
                state_data['processing'] = True
                print(f'\n⏹️ 静音 {silence_elapsed:.1f}s，开始处理...')

                wav_path = save_buffer_to_wav(state_data['audio_buffer'], sr)
                state_data['audio_buffer'] = []
                state_data['is_speaking'] = False
                state_data['speech_detected'] = False
                state_data['silence_start'] = None

                if wav_path is None:
                    state_data['processing'] = False
                    return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Audio too short, please try again...'

                pipeline_start = time.time()

                user_text = speech_to_text(wav_path)
                # ✅ 临时文件清理
                try: os.remove(wav_path)
                except: pass

                if not user_text:
                    state_data['processing'] = False
                    return state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Could not hear clearly, please repeat...'

                # ✅ 多句并行 TTS 流水线
                ai_text, audio_out = stream_llm_with_parallel_tts(
                    user_text, state_data.get('chat_history', [])
                )

                total_latency = time.time() - pipeline_start
                print(f'⏱️ 总延迟: {total_latency:.1f}s (ASR + LLM + 多句并行TTS)')

                history = state_data.get('chat_history', [])
                history.append((user_text, ai_text))
                state_data['chat_history'] = history
                state_data['processing'] = False

                return state_data, history, make_unique_audio(audio_out), f'🎙️ Done ({total_latency:.1f}s), keep speaking...'
            else:
                remaining = SILENCE_DURATION - silence_elapsed
                return state_data, state_data.get('chat_history', []), gr.update(), f'🤫 Paused... ({remaining:.1f}s)'
        else:
            s = '🎙️ Waiting for you to speak...' if _llm_ready.is_set() else '⏳ LLM loading...'
            return state_data, state_data.get('chat_history', []), gr.update(), s

# ============================================================
# ✅ 文字输入: 流式显示 + 多句并行 TTS
# ============================================================
def handle_text_input(user_text, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }
    if not user_text or not user_text.strip():
        yield state_data, state_data.get('chat_history', []), gr.update(), '🎙️ Waiting for you to speak...', ''
        return

    # ✅ 等待预热
    if not _llm_ready.is_set():
        _llm_ready.wait(timeout=120)

    voice = _get_tts_voice(user_text)
    tts_executor = ThreadPoolExecutor(max_workers=3)
    tts_futures = []
    sentence_index = 0
    submitted_up_to = 0

    # ✅ 流式生成文字，每个 token 立即推送
    temp_history = state_data.get('chat_history', []) + [(user_text, '▌')]
    yield state_data, temp_history, gr.update(), '🤖 Generating...', ''

    partial = ''
    for partial in call_llm(user_text, state_data.get('chat_history', [])):
        temp_history = state_data.get('chat_history', []) + [(user_text, partial + '▌')]
        yield state_data, temp_history, gr.update(), '🤖 Generating...', ''

        # 检测句子边界，提交 TTS
        current_buffer = partial[submitted_up_to:]
        if (re.search(r'[.!?。！？]\s*$', current_buffer)
                and len(current_buffer.strip()) >= _MIN_SENTENCE_LEN):
            sentence = current_buffer.strip()
            future = tts_executor.submit(tts_single_sentence, sentence, sentence_index, voice)
            tts_futures.append((sentence_index, future))
            print(f'  📝 [文字] Chunk {sentence_index}: "{sentence[:60]}..." → TTS')
            sentence_index += 1
            submitted_up_to = len(partial)

    # Flush remaining text
    remaining = partial[submitted_up_to:].strip()
    if remaining and len(remaining) >= 2:
        future = tts_executor.submit(tts_single_sentence, remaining, sentence_index, voice)
        tts_futures.append((sentence_index, future))
        print(f'  📝 [文字] Chunk {sentence_index} (final): "{remaining[:60]}..."')

    # ✅ 文字立刻显示完成（移除光标）
    history = state_data.get('chat_history', [])
    history.append((user_text, partial))
    state_data['chat_history'] = history
    final_history = history.copy()
    yield state_data, final_history, gr.update(), '⏳ Synthesizing voice...', ''

    # 收集 TTS 结果
    tts_futures.sort(key=lambda x: x[0])
    audio_paths = []
    for idx, future in tts_futures:
        try:
            _, audio_path = future.result(timeout=30)
            if audio_path:
                audio_paths.append(audio_path)
        except Exception as e:
            print(f'  ⚠️ TTS chunk {idx} failed: {e}')

    tts_executor.shutdown(wait=False)

    final_audio = concat_wav_files(audio_paths)

    # ✅ 清理临时 chunk 文件
    for p in audio_paths:
        if p and p != final_audio:
            try: os.remove(p)
            except: pass

    yield state_data, final_history, make_unique_audio(final_audio), '🎙️ AI replied, you can speak now...', ''


# ============================================================
# Gradio 界面 (完美对齐 + 渐变标题 + 深夜模式)
# ============================================================

custom_css = """
/* 隐藏 Gradio 顶部的默认留白和页脚水印 */
footer { display: none !important; }
.gradio-container { border: none !important; background: transparent !important; }

/* 基础背景 (白天模式) - 保留你喜欢的弥散渐变 */
body, .gradio-container {
    background-color: #fafafa !important;
    background-image: 
        radial-gradient(circle at 50% 100%, rgba(244, 114, 182, 0.25) 0%, transparent 50%),
        radial-gradient(circle at 0% 0%, rgba(167, 139, 250, 0.15) 0%, transparent 50%) !important;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
    transition: background-color 0.4s ease, background-image 0.4s ease !important;
}

body.dark-mode, body.dark-mode .gradio-container {
    background-color: #0f172a !important;
    background-image: 
        radial-gradient(circle at 50% 100%, rgba(190, 24, 93, 0.2) 0%, transparent 50%),
        radial-gradient(circle at 0% 0%, rgba(76, 29, 149, 0.2) 0%, transparent 50%) !important;
}

.ui-card {
    background: rgba(255, 255, 255, 0.85) !important;
    border-radius: 24px !important;
    box-shadow: 0 10px 40px -10px rgba(0, 0, 0, 0.08) !important;
    border: 1px solid rgba(255, 255, 255, 1) !important;
    padding: 20px !important;
    margin-bottom: 15px !important;
    transition: background 0.4s ease, border-color 0.4s ease !important;
}
body.dark-mode .ui-card {
    background: rgba(30, 41, 59, 0.85) !important;
    border: 1px solid rgba(255, 255, 255, 0.05) !important;
    box-shadow: 0 10px 40px -10px rgba(0, 0, 0, 0.4) !important;
}

.dyn-text { color: #475569 !important; transition: color 0.4s ease; }
body.dark-mode .dyn-text { color: #cbd5e1 !important; }
.dyn-subtext { color: #64748b !important; }
body.dark-mode .dyn-subtext { color: #94a3b8 !important; }

.gradient-title {
    font-size: 2.8rem !important;
    font-weight: 800 !important;
    background: linear-gradient(90deg, #c026d3, #8b5cf6) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    margin-bottom: 0 !important;
}
body.dark-mode .gradient-title { background: linear-gradient(90deg, #e879f9, #a78bfa) !important; -webkit-background-clip: text !important; -webkit-text-fill-color: transparent !important; }

.subtitle {
    font-size: 1.2rem !important;
    font-weight: 600 !important;
    color: #334155 !important;
    margin-top: 5px !important;
}
body.dark-mode .subtitle { color: #e2e8f0 !important; }

.input-row {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
    padding: 10px 15px !important;
}
.input-box { flex-grow: 1 !important; margin: 0 !important; }
.input-box > div { border: none !important; box-shadow: none !important; background: transparent !important; }
.input-box textarea {
    border-radius: 12px !important;
    border: 1px solid #cbd5e1 !important;
    padding: 12px 15px !important;
    font-size: 15px !important;
    transition: all 0.3s ease !important;
}
body.dark-mode .input-box textarea { background: #1e293b !important; border-color: #475569 !important; color: #f8fafc !important; }
.input-box textarea:focus { border-color: #8b5cf6 !important; outline: none !important; }

.send-btn {
    margin: 0 !important;
    height: 48px !important;
    border-radius: 12px !important;
    background: linear-gradient(90deg, #ec4899, #8b5cf6) !important;
    color: white !important;
    font-weight: 600 !important;
    border: none !important;
    min-width: 140px !important;
    transition: transform 0.2s ease, box-shadow 0.2s ease !important;
}
.send-btn:hover { transform: translateY(-2px); box-shadow: 0 8px 20px -5px rgba(236, 72, 153, 0.4) !important; }

.theme-btn {
    border-radius: 12px !important;
    background: transparent !important;
    border: 1px solid #cbd5e1 !important;
    color: #475569 !important;
    transition: all 0.3s ease !important;
}
body.dark-mode .theme-btn { border-color: #475569 !important; color: #cbd5e1 !important; background: rgba(30, 41, 59, 0.5) !important; }

#chatbot { background: transparent !important; border: none !important; }
#chatbot .message { border-radius: 16px !important; }
"""

theme = gr.themes.Soft(
    primary_hue="pink", 
    secondary_hue="purple",
    neutral_hue="slate"
)

with gr.Blocks(title='HWU Voice Call', css=custom_css, theme=theme) as demo:
    
    with gr.Row():
        with gr.Column(scale=9):
            gr.HTML(f"""
                <div style="text-align: center; margin-top: 2rem; margin-bottom: 2.5rem;">
                    <h1 class="gradient-title">HWU AI Student Services</h1>
                    <h2 class="subtitle">Live Voice Call</h2>
                    <p class="dyn-subtext" style="font-size: 1.05rem; max-width: 650px; margin: 1.5rem auto 0; line-height: 1.6;">
                        <strong>Instructions:</strong> Click the microphone to start the call, speak directly, and the AI will automatically reply after a {SILENCE_DURATION}s pause.<br>
                        <span style="font-size: 0.85em; opacity: 0.8;">Environment: GPU={torch.cuda.is_available()} | Whisper={WHISPER_MODEL_SIZE} | Device={device} | MultiSentenceTTS=ON</span>
                    </p>
                </div>
            """)
        with gr.Column(scale=1):
            theme_btn = gr.Button("🌓 Theme", elem_classes="theme-btn")
            theme_btn.click(None, None, None, js="""
                function() {
                    document.body.classList.toggle('dark-mode');
                }
            """)

    call_data = gr.State(value={
        'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
        'speech_detected': False, 'chat_history': [], 'processing': False
    })

    with gr.Row():
        with gr.Column(scale=3, elem_classes="ui-card"):
            chatbot = gr.Chatbot(label='Conversation History', height=420, elem_id="chatbot", type='tuples')
            
        with gr.Column(scale=1, elem_classes="ui-card"):
            status = gr.Textbox(value='⏳ LLM loading...', label='Call Status', interactive=False)
            audio_player = gr.Audio(label='🔊 AI Response', autoplay=True, interactive=False, type='filepath')
            
            gr.Markdown('<hr style="margin: 15px 0; border-color: #e2e8f0;">')
            
            mic = gr.Audio(
                sources=['microphone'], streaming=True,
                label='🎤 Microphone (Click to start)'
            )
            clear_btn = gr.Button('🗑️ Clear Conversation', variant='stop')

    with gr.Row(elem_classes="ui-card input-row"):
        text_input = gr.Textbox(
            show_label=False,
            container=False,
            placeholder='Or type your questions here...',
            scale=4,
            elem_classes="input-box"
        )
        send_btn = gr.Button('Send Message', scale=1, elem_classes="send-btn")

    # --- 核心事件绑定 ---
    mic.stream(
        fn=process_streaming_audio,
        inputs=[mic, call_data],
        outputs=[call_data, chatbot, audio_player, status]
    )
    send_btn.click(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )
    text_input.submit(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )

    def clear_all():
        return (
            {'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
             'speech_detected': False, 'chat_history': [], 'processing': False},
            [], None, '🎙️ Session Cleared'
        )
    clear_btn.click(fn=clear_all, outputs=[call_data, chatbot, audio_player, status])

print('\n🚀 启动实时语音通话 (多句并行TTS)...')

# 先测试 share 服务器是否可达
_use_share = False
try:
    import urllib.request
    urllib.request.urlopen('https://api.gradio.app/', timeout=10)
    _use_share = True
    print('✅ Gradio share 服务器可达')
except Exception as _e:
    print(f'⚠️ Gradio share 服务器不可达 ({_e})，使用 local 模式')

if _use_share:
    demo.launch(debug=True, share=True)
else:
    demo.launch(debug=True, share=False, server_name='0.0.0.0', server_port=7860)
    print('💡 如在 Kaggle，可在右侧面板查看 Web Preview (端口 7860)')


🖥️ Whisper 设备: cuda
✅ Whisper base 已加载 (GPU)
🔥 [后台] 预热 LLM...


/tmp/ipykernel_55/1914406179.py:965: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='HWU Voice Call', css=custom_css, theme=theme) as demo:
/tmp/ipykernel_55/1914406179.py:965: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title='HWU Voice Call', css=custom_css, theme=theme) as demo:
/tmp/ipykernel_55/1914406179.py:994: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  chatbot = gr.Chatbot(label='Conversation History', height=420, elem_id="chatbot", type='tuples')
/tmp/ipykernel_55/1914406179.py:994: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be cha


🚀 启动实时语音通话 (多句并行TTS)...
✅ Gradio share 服务器可达
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://a2bf9e664c03bc3870.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from /root/.ollama/models/blobs/sha256-dde5aa3fc5ffc17176b5e8bdc82f587b24b2678c6c66101bf7da77af9f7ccdff (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv   6:                               general.tags arr[str,6]   

load: printing all EOG tokens:
load:   - 128001 ('<|end_of_text|>')
load:   - 128008 ('<|eom_id|>')
load:   - 128009 ('<|eot_id|>')
load: special tokens cache size = 256
load: token to piece cache size = 0.7999 MB
print_info: arch             = llama
print_info: vocab_only       = 1
print_info: no_alloc         = 0
print_info: model type       = ?B
print_info: model params     = 3.21 B
print_info: general.name     = Llama 3.2 3B Instruct
print_info: vocab type       = BPE
print_info: n_vocab          = 128256
print_info: n_merges         = 280147
print_info: BOS token        = 128000 '<|begin_of_text|>'
print_info: EOS token        = 128009 '<|eot_id|>'
print_info: EOT token        = 128009 '<|eot_id|>'
print_info: EOM token        = 128008 '<|eom_id|>'
print_info: LF token         = 198 'Ċ'
print_info: EOG token        = 128001 '<|end_of_text|>'
print_info: EOG token        = 128008 '<|eom_id|>'
print_info: EOG token        = 128009 '<|eot_id|>'
print_info: max token length = 256
llam

[GIN] 2026/03/11 - 16:26:08 | 200 |   2.44739181s |             ::1 | POST     "/api/chat"
✅ [后台] LLM 预热完成

⏹️ 静音 1.6s，开始处理...
🎤 识别 (0.34s): Can you tell me about something about facilities in Edinburgh campus and please also provide me some links?
🔍 RAG (0.01s) folders=all → 5 docs, 2 links


time=2026-03-11T16:26:42.791Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 46135"
time=2026-03-11T16:26:43.188Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 37055"
time=2026-03-11T16:26:43.332Z level=INFO source=server.go:430 msg="starting runner" cmd="/usr/local/bin/ollama runner --ollama-engine --port 42759"
llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from /root/.ollama/models/blobs/sha256-dde5aa3fc5ffc17176b5e8bdc82f587b24b2678c6c66101bf7da77af9f7ccdff (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                         

  📝 Chunk 0: "I'd be happy to tell you about the facilities on the Edinbur..." → TTS submitted
  📝 Chunk 1: "You can find more about the various facilities on the [Campu..." → TTS submitted
  📝 Chunk 2: "One of the notable facilities on campus is the [Oriam](https..." → TTS submitted
  📝 Chunk 3: "Oriam is a state-of-the-art sports and fitness facility that..." → TTS submitted
  📝 Chunk 4: "Additionally, you can also find the [Students' Union](https:..." → TTS submitted
  📝 Chunk 5: "The [Chaplaincy](https://www.hw.ac.uk/campuses/edinburgh/fac..." → TTS submitted
  📝 Chunk 6: "If you're looking for more information about the facilities ..." → TTS submitted
  📝 Chunk 7: "Let me know if you have any other questions or if there's an..." → TTS submitted
[GIN] 2026/03/11 - 16:26:51 | 200 |    8.5098746s |             ::1 | POST     "/api/chat"
🤖 LLM (8.5s): I'd be happy to tell you about the facilities on the Edinburgh campus of Heriot-Watt University.

Yo...
🤖 LLM done, 8 TTS chunks, waiti